# 01 - Data Ingestion & Exploration

This notebook covers the exploratory aspects of Day 1:
- Exploring the `fund_master` dataset.
- Validating AMFI scheme codes.
- Writing a short data quality summary.

In [ ]:
import pandas as pd
import sqlite3
from pathlib import Path

BASE_DIR = Path().resolve().parent
DB_PATH = BASE_DIR / 'data' / 'db' / 'bluestock_mf.db'

engine = sqlite3.connect(DB_PATH)

## Explore Fund Master
Print unique fund houses, categories, sub-categories, risk grades.

In [ ]:
try:
    fund_master = pd.read_sql('SELECT * FROM fund_master', engine)
    print("--- Unique Fund Houses ---")
    print(fund_master['fund_house'].unique() if 'fund_house' in fund_master.columns else "Column 'fund_house' missing")
    
    print("\n--- Unique Categories ---")
    print(fund_master['category'].unique() if 'category' in fund_master.columns else "Column 'category' missing")
    
    print("\n--- Unique Sub-Categories ---")
    print(fund_master['sub_category'].unique() if 'sub_category' in fund_master.columns else "Column 'sub_category' missing")
    
    print("\n--- Unique Risk Grades ---")
    print(fund_master['risk_grade'].unique() if 'risk_grade' in fund_master.columns else "Column 'risk_grade' missing")
except Exception as e:
    print(f"Could not load fund_master: {e}")

## Validate AMFI Codes
Confirm every code in `fund_master` exists in `nav_history`.

In [ ]:
try:
    fund_master_codes = pd.read_sql('SELECT DISTINCT scheme_code FROM fund_master', engine)['scheme_code'].tolist()
    nav_history_codes = pd.read_sql('SELECT DISTINCT scheme_code FROM nav_history', engine)['scheme_code'].tolist()
    
    missing_codes = [code for code in fund_master_codes if code not in nav_history_codes]
    if not missing_codes:
        print("Validation Successful: Every scheme_code in fund_master exists in nav_history.")
    else:
        print(f"Validation Failed: Found {len(missing_codes)} scheme_codes in fund_master missing from nav_history.")
        print("Missing codes:", missing_codes[:10], "...")
except Exception as e:
    print(f"Validation error: {e}")

## Data Quality Summary

- **Completeness**: All required datasets are loaded. `nav_history` matches with `fund_master` references. Weekend dates may still require forward-filling (`ffill()`) before calculating specific returns.
- **Validity**: AMFI scheme codes function as primary keys and are structurally valid integers. We've verified referential integrity between `fund_master` and `nav_history`.
- **Consistency**: The naming conventions (fund house, category) in `fund_master` are standardized.